# 05 — Sonuçların Analizi ve Karşılaştırma

**Amaç:** Tüm deneylerin sonuçlarını görselleştirmek ve literatürle karşılaştırmak.

**Not:** Bu notebook kendi başına çalışır — 04'e bağımlı değildir.
Tüm modeller bu notebook içinde yeniden eğitilir.

## 0. Colab Kurulumu

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/phishing-url-detection'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/erenterakye/phishing-url-detection.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull

    %cd {REPO_DIR}
    !pip install -q -r requirements.txt

    DATA_PATH = '/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv'
else:
    REPO_DIR = '..'
    DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'

sys.path.insert(0, REPO_DIR)
print(f'IN_COLAB={IN_COLAB}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import full_preprocessing_pipeline
from src.feature_selection import rf_feature_importance, get_top5_vajrobol
from src.models import get_baseline_models, train_model
from src.evaluation import (
    evaluate_model, plot_confusion_matrix, plot_roc_curves,
    plot_feature_importance, compare_models_table
)

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

## 1. Veri ve Özellik Setleri

In [ ]:
X_train, X_test, y_train, y_test, tld_encoder, scaler = full_preprocessing_pipeline(DATA_PATH)
feature_names = list(X_train.columns)

vajrobol_5 = [f for f in get_top5_vajrobol() if f in feature_names]

print('RF Top-20 hesaplanıyor (birkaç dakika)...')
rf_top20, _ = rf_feature_importance(X_train, y_train, top_n=20)

print(f'\nTam özellik: {len(feature_names)} | RF Top-20: {len(rf_top20)} | Vajrobol-5: {len(vajrobol_5)}')

## 2. Tüm Deneyleri Çalıştır

In [ ]:
def run_experiment(X_tr, X_te, y_tr, y_te, label):
    print(f'\n=== {label} ===')
    results = {}
    trained = {}
    for name, model in get_baseline_models().items():
        model, _ = train_model(model, X_tr, y_tr)
        results[name] = evaluate_model(model, X_te, y_te)
        trained[name] = model
        m = results[name]
        print(f'  {name:<22} Acc={m["Accuracy"]:.4f} F1={m["F1-Score"]:.4f} Recall={m["Recall"]:.4f}')
    return compare_models_table(results), trained

df_exp1, trained_exp1 = run_experiment(X_train, X_test, y_train, y_test, 'Deney 1: Tam Özellik')
df_exp2, trained_exp2 = run_experiment(X_train[vajrobol_5], X_test[vajrobol_5], y_train, y_test, 'Deney 2: Vajrobol-5')
df_exp3, trained_exp3 = run_experiment(X_train[rf_top20],   X_test[rf_top20],   y_train, y_test, 'Deney 3: RF Top-20')

## 3. Confusion Matrix — En İyi Model (Deney 1)

In [ ]:
best_name = df_exp1.index[0]
y_pred_best = trained_exp1[best_name].predict(X_test)
plot_confusion_matrix(y_test, y_pred_best, f'{best_name} (Tam Özellik)')

## 4. ROC Eğrileri — Tüm Deneyler

In [ ]:
print('Deney 1 — Tam Özellik:')
plot_roc_curves(trained_exp1, X_test, y_test)

print('Deney 2 — Vajrobol-5:')
plot_roc_curves(trained_exp2, X_test[vajrobol_5], y_test)

print('Deney 3 — RF Top-20:')
plot_roc_curves(trained_exp3, X_test[rf_top20], y_test)

## 5. Özellik Önemi — RF ve XGBoost (Deney 1)

In [ ]:
print('Random Forest — Top-20 Özellik Önemi:')
plot_feature_importance(trained_exp1['Random Forest'], feature_names, top_n=20)

print('XGBoost — Top-20 Özellik Önemi:')
plot_feature_importance(trained_exp1['XGBoost'], feature_names, top_n=20)

## 6. Deney × Metrik Karşılaştırma Grafikleri

In [ ]:
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_order = df_exp1.index.tolist()

fig, axes = plt.subplots(1, len(metrics_cols), figsize=(22, 5))
for ax, metric in zip(axes, metrics_cols):
    if metric not in df_exp1.columns:
        continue
    x = np.arange(len(model_order))
    w = 0.25
    ax.bar(x - w, df_exp1[metric].fillna(0).reindex(model_order).values, w, label='Exp1: Tam',       color='steelblue')
    ax.bar(x,     df_exp2[metric].fillna(0).reindex(model_order).values, w, label='Exp2: Vajrobol5', color='tomato')
    ax.bar(x + w, df_exp3[metric].fillna(0).reindex(model_order).values, w, label='Exp3: RF Top20',  color='seagreen')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ', '\n') for m in model_order], fontsize=7)
    ax.set_ylim(0.85, 1.02)
    ax.axhline(y=0.97, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_title(metric, fontsize=10)
    ax.legend(fontsize=6)

plt.suptitle('Deney × Metrik Karşılaştırması  (kırmızı kesik = hedef %97)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Literatür ile Karşılaştırma Tablosu

In [ ]:
best_name_exp1 = df_exp1.index[0]
best_m = df_exp1.loc[best_name_exp1]
lr_vajrobol_acc  = df_exp2.loc['Logistic Regression', 'Accuracy']  if 'Logistic Regression' in df_exp2.index else None
lr_vajrobol_f1   = df_exp2.loc['Logistic Regression', 'F1-Score']  if 'Logistic Regression' in df_exp2.index else None
lr_vajrobol_rec  = df_exp2.loc['Logistic Regression', 'Recall']    if 'Logistic Regression' in df_exp2.index else None

literature = pd.DataFrame([
    {'Kaynak': 'Prasad & Chandra (2024)', 'Yöntem': 'BernoulliNB+PAC+SGD',    'Veri Seti': 'PhiUSIIL',     'Accuracy': 0.9924, 'F1': 0.9921, 'Recall': '—'},
    {'Kaynak': 'Vajrobol vd. (2024)',     'Yöntem': 'LR (5 özellik)',          'Veri Seti': 'PhiUSIIL',     'Accuracy': 0.9997, 'F1': 0.9997, 'Recall': '—'},
    {'Kaynak': 'Yoon vd. (2024)',         'Yöntem': 'CNN+Transformer+GCN',     'Veri Seti': 'Common Crawl', 'Accuracy': 0.9812, 'F1': 0.9789, 'Recall': '—'},
    {'Kaynak': 'Rao vd. (2025)',          'Yöntem': 'Super Learner Ensemble',  'Veri Seti': 'PhishDump',    'Accuracy': 0.9893, 'F1': '—',    'Recall': '—'},
    {'Kaynak': 'Taha vd. (2024)',         'Yöntem': 'LR,DT,RF,XGB',           'Veri Seti': '11K Phishing', 'Accuracy': 0.9689, 'F1': '—',    'Recall': '—'},
    {'Kaynak': f'Bu Çalışma — {best_name_exp1} (Exp1)',
     'Yöntem': best_name_exp1, 'Veri Seti': 'PhiUSIIL',
     'Accuracy': round(best_m['Accuracy'], 4),
     'F1': round(best_m['F1-Score'], 4),
     'Recall': round(best_m['Recall'], 4)},
    {'Kaynak': 'Bu Çalışma — LR (Vajrobol-5)',
     'Yöntem': 'LR (5 özellik)', 'Veri Seti': 'PhiUSIIL',
     'Accuracy': round(lr_vajrobol_acc, 4) if lr_vajrobol_acc else '—',
     'F1': round(lr_vajrobol_f1, 4) if lr_vajrobol_f1 else '—',
     'Recall': round(lr_vajrobol_rec, 4) if lr_vajrobol_rec else '—'},
]).set_index('Kaynak')

print('=== Literatür Karşılaştırması ===')
print(literature.to_string())

## 8. Hedef Metrik Kontrolü

In [ ]:
targets = {'Accuracy': 0.97, 'F1-Score': 0.96, 'Recall': 0.97, 'ROC-AUC': 0.98}

print('=== Hedef Metrik Kontrolü (Deney 1) ===')
for col, threshold in targets.items():
    if col not in df_exp1.columns:
        continue
    passing = df_exp1[df_exp1[col] >= threshold].index.tolist()
    print(f'{col} >= {threshold}: {passing if passing else "HİÇBİRİ"}')